# Fetch Ingredients from RxNorm

## Check Python Version

In [1]:
!python --version

Python 3.11.17


## Setup Config

In [2]:
PROJECT_DIR = "../../"

import pathlib
OUTPUT_DIR = pathlib.Path(PROJECT_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

FILE_RX_NORM_INGREDIENTS = "rxnorm/rx_norm_ingredients.csv"
RX_NORM_INGREDIENTS_PATH = OUTPUT_DIR.joinpath(FILE_RX_NORM_INGREDIENTS)
print(f"All Drug RxNorm Ingredients Location: {RX_NORM_INGREDIENTS_PATH}")

Dataset Directory: ../..
All Drug RxNorm Ingredients Location: ../../rxnorm/rx_norm_ingredients.csv


## Create RxNormConceptFetcher

In [3]:
import time
import requests

class RxNormConceptFetcher:
    BASE_URL = "https://rxnav.nlm.nih.gov/REST"

    def _get_allconcepts(self, tty: str):
        try:
            url = f"{self.BASE_URL}/allconcepts.json"
    
            response = requests.get(
                url,
                params={"tty": tty},
                timeout=20
            )
    
            response.raise_for_status()
            data = response.json()
    
            concepts = (
                data.get("minConceptGroup", {})
                .get("minConcept", [])
            )
    
            if not concepts:
                return None

            results = []
            for item in concepts:
                if item.get("tty") != tty:
                    continue
                    
                rx_cui = item.get("rxcui")
                ingredient_name = item.get("name")
                
                results.append({
                    "rx_cui": rx_cui,
                    "name": ingredient_name
                })
                
            return results
        except Exception as e:
            print(f"[ALLCONCEPTS ERROR] {tty}: {e}")
            return None

    def get_all_ingredients(self):
        tty = "IN"
            
        return self._get_allconcepts(tty)

## Fetch All Ingredients in RxNorm

In [4]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


fetcher = RxNormConceptFetcher()
ingredients = fetcher.get_all_ingredients()
ingredients_df = pd.DataFrame(ingredients)
ingredients_df.to_csv(RX_NORM_INGREDIENTS_PATH, index=False)

print(f"Total rows: {ingredients_df.shape[0]}\n")
print(f"Ingredients successfully stored at {RX_NORM_INGREDIENTS_PATH}")

Total rows: 14718

Ingredients successfully stored at ../../rxnorm/rx_norm_ingredients.csv
